### Training pipeline

- This notebook fine-tunes the English-only training model and writes prediction score files for downstream workflow analysis.
- It expects prepared split CSV files under `./data`.
- See the README for dataset preparation instructions.

In [ ]:
# Cell 1: setup + config

import os
import json
import math
import random
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from sklearn.metrics import accuracy_score, precision_recall_fscore_support
from sklearn.model_selection import train_test_split
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    EarlyStoppingCallback,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 300)

CONFIG = {
    "seed": 42,
    "model_name": "google-bert/bert-base-multilingual-cased",
    "data_root": "./data",
    "output_root": "./outputs",
    "files": {
        "train_master": "splits/train_master.csv",
        "test_master": "splits/test_master.csv",
        "train_english": "splits/train_english.csv",
        "test_english": "splits/test_english.csv",
        "test_codemix": "splits/test_codemix.csv",
        "test_tamil": "splits/test_tamil.csv"
    },
    "internal_eval": {
        "enabled": True,
        "eval_from_train_size": 0.20
    },
    "training": {
        "num_labels": 2,
        "max_length": 320,
        "num_train_epochs": 4,
        "learning_rate": 1.5e-5,
        "train_batch_size": 8,
        "eval_batch_size": 16,
        "weight_decay": 0.01,
        "warmup_ratio": 0.1,
        "logging_steps": 25,
        "save_total_limit": 2,
        "early_stopping_patience": 2,
        "metric_for_best_model": "macro_f1",
        "greater_is_better": True,
        "fp16": torch.cuda.is_available()
    },
    "smoke_test": {
        "enabled": False,
        "train_cap": 200,
        "eval_cap": 50,
        "test_cap": 100,
        "epochs": 1
    }
}

def set_all_seeds(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)

set_all_seeds(CONFIG["seed"])
DATA_ROOT = Path(CONFIG["data_root"])
ROOT = Path(CONFIG["output_root"])
DIRS = {
    "checkpoints": ROOT / "checkpoints" / "english",
    "predictions": ROOT / "predictions",
    "logs": ROOT / "logs"
}
for d in DIRS.values():
    d.mkdir(parents=True, exist_ok=True)

if CONFIG["smoke_test"]["enabled"]:
    CONFIG["training"]["num_train_epochs"] = CONFIG["smoke_test"]["epochs"]

print("Data root  :", DATA_ROOT)
print("Output root:", ROOT)
print("Device     :", "cuda" if torch.cuda.is_available() else "cpu")

<!-- NOTE: Generated artifacts (checkpoints, predictions, logs) are written to `./outputs` during execution but are not included in the public repository by default. -->

In [ ]:
def read_split_csv(relative_path):
    path = DATA_ROOT / relative_path
    if not path.exists():
        raise FileNotFoundError(
            f"Missing required file: {path}. See the README for dataset preparation and input file instructions."
        )
    return pd.read_csv(path)


def basic_view_check(df, name):
    required = ["sourceid", "label", "text", "view_name"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"{name} missing columns: {missing}")
    out = df.copy()
    out["label"] = out["label"].astype(int)
    out["text"] = out["text"].fillna("").astype(str).str.strip()
    out = out[out["text"] != ""].reset_index(drop=True)
    return out


def make_view_df(df, text_col, view_name):
    keep_cols = [c for c in ["sourceid", "label", "textenglish", "textcodemix", "source_dataset"] if c in df.columns]
    out = df[keep_cols].copy()
    out["text"] = df[text_col].astype(str).str.strip()
    out["view_name"] = view_name
    return out.reset_index(drop=True)


def maybe_cap(df, n, seed=42):
    if n is None or len(df) <= n:
        return df.reset_index(drop=True)
    return df.sample(n=n, random_state=seed).reset_index(drop=True)


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(labels, preds, average="macro", zero_division=0)
    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
        "macro_precision": mp,
        "macro_recall": mr,
        "macro_f1": mf1,
    }


def tokenize_batch(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding=False,
        max_length=CONFIG["training"]["max_length"]
    )


def dataframe_to_dataset(df):
    keep_cols = [c for c in ["sourceid", "label", "text", "textenglish", "textcodemix", "source_dataset", "view_name"] if c in df.columns]
    ds = Dataset.from_pandas(df[keep_cols].copy(), preserve_index=False)
    ds = ds.map(tokenize_batch, batched=True)
    ds = ds.rename_column("label", "labels")
    ds.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])
    return ds


def predict_scores(trainer, df, scenario_name):
    ds = dataframe_to_dataset(df)
    pred_output = trainer.predict(ds)
    logits = pred_output.predictions
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    pred_labels = probs.argmax(axis=1)
    out = df.copy().reset_index(drop=True)
    out["prob_non_hate"] = probs[:, 0]
    out["prob_hate"] = probs[:, 1]
    out["pred_label"] = pred_labels
    out["scenario"] = scenario_name
    return out


def save_json(obj, path):
    with open(path, "w") as f:
        json.dump(obj, f, indent=2)

In [ ]:
# Cell 3: load data
# Note: train on english only, but test on english,tamil and codemix views.

train_master = read_split_csv(CONFIG["files"]["train_master"])
test_master = read_split_csv(CONFIG["files"]["test_master"])
train_english = basic_view_check(read_split_csv(CONFIG["files"]["train_english"]), "train_english")
test_english = basic_view_check(read_split_csv(CONFIG["files"]["test_english"]), "test_english")
test_codemix = basic_view_check(read_split_csv(CONFIG["files"]["test_codemix"]), "test_codemix")
test_tamil = basic_view_check(read_split_csv(CONFIG["files"]["test_tamil"]), "test_tamil")

if not CONFIG["internal_eval"]["enabled"]:
    raise ValueError("This v3 notebook requires internal_eval.enabled=True because val_english.csv is no longer saved.")

train_sub_master, eval_sub_master = train_test_split(
    train_master,
    test_size=CONFIG["internal_eval"]["eval_from_train_size"],
    random_state=CONFIG["seed"],
    stratify=train_master["label"]
)
train_sub_master = train_sub_master.reset_index(drop=True)
eval_sub_master = eval_sub_master.reset_index(drop=True)

train_sub_english = basic_view_check(make_view_df(train_sub_master, "textenglish", "train_english_internal"), "train_sub_english")
eval_sub_english = basic_view_check(make_view_df(eval_sub_master, "textenglish", "english_dev_internal"), "eval_sub_english")

if CONFIG["smoke_test"]["enabled"]:
    train_sub_english = maybe_cap(train_sub_english, CONFIG["smoke_test"]["train_cap"], CONFIG["seed"])
    eval_sub_english = maybe_cap(eval_sub_english, CONFIG["smoke_test"]["eval_cap"], CONFIG["seed"])
    test_english = maybe_cap(test_english, CONFIG["smoke_test"]["test_cap"], CONFIG["seed"])
    test_codemix = maybe_cap(test_codemix, CONFIG["smoke_test"]["test_cap"], CONFIG["seed"])
    test_tamil = maybe_cap(test_tamil, CONFIG["smoke_test"]["test_cap"], CONFIG["seed"])

print("train_master      :", train_master.shape)
print("test_master       :", test_master.shape)
print("train_sub_english :", train_sub_english.shape)
print("eval_sub_english  :", eval_sub_english.shape)
print("test_english      :", test_english.shape)
print("test_codemix      :", test_codemix.shape)
print("test_tamil        :", test_tamil.shape)
print("\nTrain label distribution:")
print(train_sub_english["label"].value_counts(normalize=True))

In [ ]:
# Cell 4: tokenizer + datasets
# Note: internal clean-English eval is for checkpoint selection only.

tokenizer = AutoTokenizer.from_pretrained(CONFIG["model_name"])
data_collator = DataCollatorWithPadding(tokenizer=tokenizer, padding="longest", pad_to_multiple_of=8 if CONFIG["training"]["fp16"] else None)
train_ds = dataframe_to_dataset(train_sub_english)
eval_ds = dataframe_to_dataset(eval_sub_english)

print("Tokenizer loaded:", CONFIG["model_name"])
print("Train dataset len:", len(train_ds))
print("Eval dataset len :", len(eval_ds))

In [ ]:
# Cell 5: model + trainer
# Note: first version - english-only training.

model = AutoModelForSequenceClassification.from_pretrained(
    CONFIG["model_name"],
    num_labels=CONFIG["training"]["num_labels"]
)

num_training_steps = math.ceil(len(train_ds) / CONFIG["training"]["train_batch_size"]) * CONFIG["training"]["num_train_epochs"]
warmup_steps = int(num_training_steps * CONFIG["training"]["warmup_ratio"])

training_args = TrainingArguments(
    output_dir=str(DIRS["checkpoints"]),
    num_train_epochs=CONFIG["training"]["num_train_epochs"],
    learning_rate=CONFIG["training"]["learning_rate"],
    per_device_train_batch_size=CONFIG["training"]["train_batch_size"],
    per_device_eval_batch_size=CONFIG["training"]["eval_batch_size"],
    weight_decay=CONFIG["training"]["weight_decay"],
    warmup_steps=warmup_steps,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="steps",
    logging_steps=CONFIG["training"]["logging_steps"],
    load_best_model_at_end=True,
    metric_for_best_model=CONFIG["training"]["metric_for_best_model"],
    greater_is_better=CONFIG["training"]["greater_is_better"],
    save_total_limit=CONFIG["training"]["save_total_limit"],
    fp16=CONFIG["training"]["fp16"],
    report_to="none",
    seed=CONFIG["seed"]
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=eval_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=CONFIG["training"]["early_stopping_patience"])],
)

In [ ]:
# Cell 6: train + save checkpoint

train_result = trainer.train()
eval_result = trainer.evaluate()
trainer.save_model(str(DIRS["checkpoints"]))
tokenizer.save_pretrained(str(DIRS["checkpoints"]))

train_metrics = dict(train_result.metrics)
eval_metrics = dict(eval_result)
save_json(train_metrics, DIRS["logs"] / "train_metrics.json")
save_json(eval_metrics, DIRS["logs"] / "eval_metrics.json")

print("Best checkpoint saved to:", DIRS["checkpoints"])
print("\nInternal eval metrics:")
print(eval_metrics)

In [ ]:
# Cell 7: run predictions and save raw score files

english_dev_scores = predict_scores(trainer, eval_sub_english, "english_dev")
english_test_scores = predict_scores(trainer, test_english, "english_test")
codemix_test_scores = predict_scores(trainer, test_codemix, "codemix_test")
tamil_test_scores = predict_scores(trainer, test_tamil, "tamil_test")

english_dev_scores.to_csv(DIRS["predictions"] / "english_dev_scores.csv", index=False)
english_test_scores.to_csv(DIRS["predictions"] / "english_test_scores.csv", index=False)
codemix_test_scores.to_csv(DIRS["predictions"] / "codemix_test_scores.csv", index=False)
tamil_test_scores.to_csv(DIRS["predictions"] / "tamil_test_scores.csv", index=False)

print(f"Prediction files saved to: {DIRS['predictions']}")

In [ ]:
# Cell 8: quick summary

summary_df = pd.DataFrame([
    {
        "model_name": CONFIG["model_name"],
        "train_master_rows": len(train_master),
        "train_used_rows": len(train_sub_english),
        "internal_eval_rows": len(eval_sub_english),
        "test_master_rows": len(test_master),
        "test_english_rows": len(test_english),
        "test_codemix_rows": len(test_codemix),
        "test_tamil_rows": len(test_tamil),
        "epochs": CONFIG["training"]["num_train_epochs"],
        "max_length": CONFIG["training"]["max_length"],
        "learning_rate": CONFIG["training"]["learning_rate"],
        "checkpoint_dir": str(DIRS["checkpoints"]),
        "predictions_dir": str(DIRS["predictions"])
    }
])
summary_df.to_csv(DIRS["logs"] / "run_summary.csv", index=False)

print("Run summary saved to:", DIRS["logs"] / "run_summary.csv")

In [ ]:
# Cell 9: zero-shot helper
# Note: shared inference utility for pre-trained hate-speech models used without fine-tuning.

def predict_zeroshot_scores(zs_model_name, df, scenario_name, hate_label_id=1, batch_size=32):
    """
    Load a pre-trained (already hate-speech-fine-tuned) model and run zero-shot
    inference on `df`.  No training is performed.

    Parameters
    ----------
    zs_model_name : str   HuggingFace model id
    df            : pd.DataFrame  must contain 'text' and 'label' columns
    scenario_name : str   tag written to the 'scenario' column
    hate_label_id : int   which logit index maps to hate (default 1)
    batch_size    : int   inference batch size

    Returns
    -------
    pd.DataFrame with columns: prob_non_hate, prob_hate, pred_label, scenario
    """
    device = "cuda" if torch.cuda.is_available() else "cpu"
    zs_tok = AutoTokenizer.from_pretrained(zs_model_name)
    zs_mod = AutoModelForSequenceClassification.from_pretrained(zs_model_name)
    zs_mod.to(device)
    zs_mod.eval()

    texts = df["text"].fillna("").astype(str).str.strip().tolist()
    all_probs = []

    with torch.no_grad():
        for i in range(0, len(texts), batch_size):
            batch = texts[i : i + batch_size]
            enc = zs_tok(
                batch,
                truncation=True,
                padding=True,
                max_length=CONFIG["training"]["max_length"],
                return_tensors="pt",
            ).to(device)
            logits = zs_mod(**enc).logits
            probs = torch.softmax(logits, dim=1).cpu().numpy()
            all_probs.extend(probs)

    all_probs = np.array(all_probs)  # (N, num_labels)
    non_hate_id = 1 - hate_label_id
    raw_preds = all_probs.argmax(axis=1)

    out = df.copy().reset_index(drop=True)
    out["prob_non_hate"] = all_probs[:, non_hate_id]
    out["prob_hate"] = all_probs[:, hate_label_id]
    # normalise so pred_label is always in {0=non-hate, 1=hate}
    out["pred_label"] = (raw_preds == hate_label_id).astype(int)
    out["scenario"] = scenario_name

    # free GPU memory
    del zs_mod
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    return out


def eval_metrics_from_df(score_df):
    """Compute classification metrics from a scored dataframe."""
    labels = score_df["label"].values
    preds  = score_df["pred_label"].values
    acc = accuracy_score(labels, preds)
    p, r, f1, _   = precision_recall_fscore_support(labels, preds, average="binary", zero_division=0)
    mp, mr, mf1, _ = precision_recall_fscore_support(labels, preds, average="macro",  zero_division=0)
    return {
        "accuracy":        round(float(acc), 4),
        "precision":       round(float(p),   4),
        "recall":          round(float(r),   4),
        "f1":              round(float(f1),  4),
        "macro_precision": round(float(mp),  4),
        "macro_recall":    round(float(mr),  4),
        "macro_f1":        round(float(mf1), 4),
    }


print("Zero-shot helpers ready.")


In [ ]:
# Cell 10: HateBERT Zero-shot
# Note: GroNLP/hateBERT is pre-trained on RAL-E (hateful Reddit content);
#             used here without any fine-tuning on our dataset.

HATEBERT_MODEL = "GroNLP/hateBERT"

print(f"Model : {HATEBERT_MODEL}")
print("Mode  : zero-shot (no fine-tuning on this dataset)")
print()

hatebert_dev_scores     = predict_zeroshot_scores(HATEBERT_MODEL, eval_sub_english,  "hatebert_english_dev")
hatebert_test_scores    = predict_zeroshot_scores(HATEBERT_MODEL, test_english, "hatebert_english_test")
hatebert_codemix_scores = predict_zeroshot_scores(HATEBERT_MODEL, test_codemix, "hatebert_codemix_test")
hatebert_tamil_scores   = predict_zeroshot_scores(HATEBERT_MODEL, test_tamil, "hatebert_tamil_test")

hatebert_dev_scores.to_csv(    DIRS["predictions"] / "hatebert_zeroshot_dev_scores.csv",           index=False)
hatebert_test_scores.to_csv(   DIRS["predictions"] / "hatebert_zeroshot_english_test_scores.csv",  index=False)
hatebert_codemix_scores.to_csv(DIRS["predictions"] / "hatebert_zeroshot_codemix_test_scores.csv", index=False)
hatebert_tamil_scores.to_csv(   DIRS["predictions"] / "hatebert_zeroshot_tamil_test_scores.csv",   index=False)

hatebert_metrics = {
    "model":       HATEBERT_MODEL,
    "type":        "zero_shot",
    "english_dev":  eval_metrics_from_df(hatebert_dev_scores),
    "english_test": eval_metrics_from_df(hatebert_test_scores),
    "codemix_test": eval_metrics_from_df(hatebert_codemix_scores),
    "tamil_test":  eval_metrics_from_df(hatebert_tamil_scores)
}
save_json(hatebert_metrics, DIRS["logs"] / "hatebert_zeroshot_metrics.json")

print("HateBERT Zero-shot metrics:")
for split, m in hatebert_metrics.items():
    if isinstance(m, dict):
        print(f"  {split}: macro_f1={m['macro_f1']:.4f}  accuracy={m['accuracy']:.4f}")

print(f"\nHateBERT zero-shot prediction files saved to: {DIRS['predictions']}")

In [ ]:
# Cell 11: MetaHateBERT Zero-shot
# Note: irlab-udc/MetaHateBERT is a cross-lingual hate-speech model trained
#             via meta-learning; used here without any fine-tuning on the dataset.

METAHATEBERT_MODEL = "irlab-udc/MetaHateBERT"

print(f"Model : {METAHATEBERT_MODEL}")
print("Mode  : zero-shot (no fine-tuning on this dataset)")
print()

metahatebert_dev_scores     = predict_zeroshot_scores(METAHATEBERT_MODEL, eval_sub_english,  "metahatebert_english_dev")
metahatebert_test_scores    = predict_zeroshot_scores(METAHATEBERT_MODEL, test_english, "metahatebert_english_test")
metahatebert_codemix_scores = predict_zeroshot_scores(METAHATEBERT_MODEL, test_codemix, "metahatebert_codemix_test")
metahatebert_tamil_scores   = predict_zeroshot_scores(METAHATEBERT_MODEL, test_tamil, "metahatebert_tamil_test")

metahatebert_dev_scores.to_csv(    DIRS["predictions"] / "metahatebert_zeroshot_dev_scores.csv",           index=False)
metahatebert_test_scores.to_csv(   DIRS["predictions"] / "metahatebert_zeroshot_english_test_scores.csv",  index=False)
metahatebert_codemix_scores.to_csv(DIRS["predictions"] / "metahatebert_zeroshot_codemix_test_scores.csv", index=False)
metahatebert_tamil_scores.to_csv(   DIRS["predictions"] / "metahatebert_zeroshot_tamil_test_scores.csv", index=False)

metahatebert_metrics = {
    "model":       METAHATEBERT_MODEL,
    "type":        "zero_shot",
    "english_dev":  eval_metrics_from_df(metahatebert_dev_scores),
    "english_test": eval_metrics_from_df(metahatebert_test_scores),
    "codemix_test": eval_metrics_from_df(metahatebert_codemix_scores),
    "tamil_test":  eval_metrics_from_df(metahatebert_tamil_scores)
}
save_json(metahatebert_metrics, DIRS["logs"] / "metahatebert_zeroshot_metrics.json")

print("MetaHateBERT Zero-shot metrics:")
for split, m in metahatebert_metrics.items():
    if isinstance(m, dict):
        print(f"  {split}: macro_f1={m['macro_f1']:.4f}  accuracy={m['accuracy']:.4f}")

print(f"\nMetaHateBERT zero-shot prediction files saved to: {DIRS['predictions']}")

In [ ]:
# Cell 12: three-way comparison summary
# Note: side-by-side view of mBERT (fine-tuned) vs both zero-shot baselines.

comparison_rows = []
for model_tag, dev_df, test_df, codemix_df, tamil_scores_df in [
    ("mBERT (fine-tuned)",       english_dev_scores,       english_test_scores,       codemix_test_scores,       tamil_test_scores),
    ("HateBERT (zero-shot)",     hatebert_dev_scores,      hatebert_test_scores,      hatebert_codemix_scores,     hatebert_tamil_scores),
    ("MetaHateBERT (zero-shot)", metahatebert_dev_scores,  metahatebert_test_scores,  metahatebert_codemix_scores, metahatebert_tamil_scores),
]:
    for split_name, split_df in [
        ("english_dev",  dev_df),
        ("english_test", test_df),
        ("codemix_test", codemix_df),
        ("tamil_test", tamil_scores_df)
    ]:
        m = eval_metrics_from_df(split_df)
        comparison_rows.append({"model": model_tag, "split": split_name, **m})

comparison_df = pd.DataFrame(comparison_rows)
comparison_df.to_csv(DIRS["logs"] / "model_comparison_summary.csv", index=False)

print("=== Model comparison summary ===")
display(
    comparison_df[["model", "split", "accuracy", "f1", "macro_f1"]]
    .sort_values(["split", "model"])
    .reset_index(drop=True)
)
print("Saved:", DIRS["logs"] / "model_comparison_summary.csv")


In [ ]:
# Cell 13: visualise and interpret model comparison

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd
from pathlib import Path

# ── load results
comp = pd.read_csv(DIRS["logs"] / "model_comparison_summary.csv")

MODEL_ORDER  = ["mBERT (fine-tuned)", "HateBERT (zero-shot)", "MetaHateBERT (zero-shot)"]
SPLIT_ORDER  = ["english_dev", "english_test", "codemix_test", "tamil_test"]
SPLIT_LABELS = {"english_dev": "English dev", "english_test": "English test", "codemix_test": "Codemix test", "tamil_test": "Tamil test"}
COLORS       = ["#3266ad", "#1D9E75", "#D85A30"]
METRICS      = ["macro_f1", "f1", "accuracy", "precision", "recall"]
METRIC_LABELS= {"macro_f1": "Macro F1", "f1": "Binary F1",
                "accuracy": "Accuracy", "precision": "Precision", "recall": "Recall"}

# pivot for easy slicing
pivot = comp.pivot_table(index="model", columns="split", values=METRICS)


# PLOT 1 — Grouped bar chart: Macro F1 across splits

fig1, ax1 = plt.subplots(figsize=(10, 5))

n_models = len(MODEL_ORDER)
n_splits = len(SPLIT_ORDER)
x = np.arange(n_splits)
width = 0.25
offsets = np.linspace(-(n_models - 1) / 2, (n_models - 1) / 2, n_models) * width

for i, (model, color) in enumerate(zip(MODEL_ORDER, COLORS)):
    vals = [pivot.loc[model, ("macro_f1", s)] for s in SPLIT_ORDER]
    bars = ax1.bar(x + offsets[i], vals, width=width * 0.9, color=color,
                   alpha=0.85, label=model)
    for bar, v in zip(bars, vals):
        ax1.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
                 f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax1.set_xticks(x)
ax1.set_xticklabels([SPLIT_LABELS[s] for s in SPLIT_ORDER], fontsize=11)
ax1.set_ylabel("Macro F1", fontsize=11)
ax1.set_ylim(0, 1.05)
ax1.set_title("Macro F1 by model and evaluation split", fontsize=13, pad=12)
ax1.legend(fontsize=9, loc="lower right")
ax1.spines[["top", "right"]].set_visible(False)
ax1.grid(axis="y", linestyle="--", alpha=0.4)
fig1.tight_layout()
fig1.savefig(DIRS["logs"] / "plot1_macro_f1_grouped_bar.png", dpi=150)
plt.show()
print("Saved: plot1_macro_f1_grouped_bar.png")


# PLOT 2 — Metrics heatmap per split

fig2, axes2 = plt.subplots(1, 3, figsize=(14, 4), sharey=True)

for ax, split in zip(axes2, SPLIT_ORDER[:-1]): # Exclude 'tamil_test' from this heatmap
    mat = np.array([[pivot.loc[m, (metric, split)] for metric in METRICS]
                     for m in MODEL_ORDER])
    im = ax.imshow(mat, vmin=0, vmax=1, cmap="viridis", aspect="auto")
    ax.set_xticks(range(len(METRICS)))
    ax.set_xticklabels([METRIC_LABELS[m] for m in METRICS], rotation=35, ha="right", fontsize=9)
    ax.set_yticks(range(len(MODEL_ORDER)))
    ax.set_yticklabels([m.replace(" (", "\n(") for m in MODEL_ORDER], fontsize=8)
    ax.set_title(SPLIT_LABELS[split], fontsize=11)
    for r in range(len(MODEL_ORDER)):
        for c in range(len(METRICS)):
            ax.text(c, r, f"{mat[r, c]:.3f}", ha="center", va="center",
                    fontsize=8, color="white" if mat[r, c] > 0.6 else "#333")

fig2.colorbar(im, ax=axes2[-1], fraction=0.04, pad=0.04)
fig2.suptitle("Metrics heatmap — all models × splits (excluding Tamil)", fontsize=13, y=1.02)
fig2.tight_layout()
fig2.savefig(DIRS["logs"] / "plot2_metrics_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plot2_metrics_heatmap.png")


# PLOT 3 — Radar chart (english test split)

cats  = METRICS
N     = len(cats)
angles = np.linspace(0, 2 * np.pi, N, endpoint=False).tolist()
angles += angles[:1]

fig3, ax3 = plt.subplots(figsize=(6, 6), subplot_kw=dict(polar=True))

for model, color in zip(MODEL_ORDER, COLORS):
    vals = [pivot.loc[model, (m, "english_test")] for m in cats] + \
           [pivot.loc[model, (cats[0], "english_test")]]
    ax3.plot(angles, vals, color=color, linewidth=2, label=model)
    ax3.fill(angles, vals, color=color, alpha=0.08)

ax3.set_thetagrids(np.degrees(angles[:-1]), [METRIC_LABELS[m] for m in cats], fontsize=10)
ax3.set_ylim(0, 1)
ax3.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax3.set_yticklabels(["0.2", "0.4", "0.6", "0.8", "1.0"], fontsize=7, color="gray")
ax3.set_title("Radar — English test split", fontsize=13, pad=18)
ax3.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
ax3.spines["polar"].set_linewidth(0.5)
fig3.tight_layout()
fig3.savefig(DIRS["logs"] / "plot3_radar_english_test.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved: plot3_radar_english_test.png")


# PLOT 4 — English → Codemix degradation bar chart

fig4, ax4 = plt.subplots(figsize=(8, 4))

drops = [
    pivot.loc[m, ("macro_f1", "english_test")] - pivot.loc[m, ("macro_f1", "codemix_test")]
    for m in MODEL_ORDER
]
bars4 = ax4.barh(MODEL_ORDER, drops, color=COLORS, alpha=0.85, height=0.5)
for bar, v in zip(bars4, drops):
    ax4.text(v + 0.002, bar.get_y() + bar.get_height() / 2,
             f"{v:.3f}", va="center", fontsize=10)

ax4.set_xlabel("Macro F1 drop (english test → codemix test)", fontsize=11)
ax4.set_title("Cross-lingual degradation", fontsize=13, pad=10)
ax4.spines[["top", "right"]].set_visible(False)
ax4.grid(axis="x", linestyle="--", alpha=0.4)
ax4.invert_yaxis()
fig4.tight_layout()
fig4.savefig(DIRS["logs"] / "plot4_codemix_degradation.png", dpi=150)
plt.show()
print("Saved: plot4_codemix_degradation.png")


# PLOT 5 — Grouped bar chart: Macro F1 for Tamil test split

fig5, ax5 = plt.subplots(figsize=(10, 5))

split_to_plot = "tamil_test"

x_pos = np.arange(n_models)
width = 0.5

vals_tamil = [pivot.loc[model, ("macro_f1", split_to_plot)] for model in MODEL_ORDER]
bars5 = ax5.bar(x_pos, vals_tamil, width=width, color=COLORS, alpha=0.85)

for bar, v in zip(bars5, vals_tamil):
    ax5.text(bar.get_x() + bar.get_width() / 2, v + 0.005,
             f"{v:.3f}", ha="center", va="bottom", fontsize=8)

ax5.set_xticks(x_pos)
ax5.set_xticklabels(MODEL_ORDER, fontsize=11, rotation=20, ha="right")
ax5.set_ylabel("Macro F1", fontsize=11)
ax5.set_ylim(0, 1.05)
ax5.set_title(f"Macro F1 on {SPLIT_LABELS[split_to_plot]} by Model", fontsize=13, pad=12)
ax5.spines[["top", "right"]].set_visible(False)
ax5.grid(axis="y", linestyle="--", alpha=0.4)
fig5.tight_layout()
fig5.savefig(DIRS["logs"] / "plot5_macro_f1_tamil_test_bar.png", dpi=150)
plt.show()
print(f"Saved: plot5_macro_f1_tamil_test_bar.png")


# INTERPRETATION PRINTOUT

print("\n" + "=" * 60)
print("RESULT INTERPRETATION")
print("=" * 60)

for split in SPLIT_ORDER:
    best_model = max(MODEL_ORDER, key=lambda m: pivot.loc[m, ("macro_f1", split)])
    best_score = pivot.loc[best_model, ("macro_f1", split)]
    print(f"\n[{SPLIT_LABELS[split]}]")
    print(f"  Best model : {best_model} (Macro F1 = {best_score:.4f})")
    for m in MODEL_ORDER:
        mf1 = pivot.loc[m, ("macro_f1", split)]
        acc  = pivot.loc[m, ("accuracy",  split)]
        f1   = pivot.loc[m, ("f1",        split)]
        print(f"  {m:<35} macro_f1={mf1:.4f}  acc={acc:.4f}  f1={f1:.4f}")

print("\n[Cross-lingual degradation (Macro F1 drop: english_test → codemix_test)]")
for m, d in zip(MODEL_ORDER, drops):
    print(f"  {m:<35} drop = {d:.4f}  ({'+' if d>0 else ''}{d*100:.1f} pp)")

best_transfer = MODEL_ORDER[np.argmin(drops)]
print(f"\n  => '{best_transfer}' shows the best cross-lingual transfer (smallest drop).")

print("\n[Fine-tuning gain: mBERT vs best zero-shot on english_test]")
zs_best = max(["HateBERT (zero-shot)", "MetaHateBERT (zero-shot)"],
               key=lambda m: pivot.loc[m, ("macro_f1", "english_test")])
gain = (pivot.loc["mBERT (fine-tuned)", ("macro_f1", "english_test")] -
        pivot.loc[zs_best, ("macro_f1", "english_test")])
print(f"  mBERT (fine-tuned) vs {zs_best}: +{gain:.4f} macro F1")
print("=" * 60)

In [ ]:
# Cell 14: Final message (Colab-specific runtime disconnect code removed)
print("Run complete.")